In [3]:
from xlmexlab.parser_nanoparticles import ParserNanoparticle

_nanoparticles_parser = ParserNanoparticle()

text = """318.3±0.25 | nm | Conj-HSLN | not extractable"""
parameters = "size_nm"
print (_nanoparticles_parser.parse_response(text, parameters))

{'size_nm': [{'value': '318.3±0.25', 'unit': 'nm', 'drug_name': 'Conj-HSLN', 'size_type': 'not extractable'}]}


In [2]:
# Source - https://stackoverflow.com/q/8200342
# Posted by Shamika, modified by community. See post 'Timeline' for change history
# Retrieved 2026-07-10, License - CC BY-SA 3.0

a = ["asd","def","ase","dfg","asd","def","dfg"]
# Source - https://stackoverflow.com/a/8200352
# Posted by Krumelur
# Retrieved 2026-07-10, License - CC BY-SA 3.0

a = list(dict.fromkeys(a))
print (a)

['asd', 'def', 'ase', 'dfg']


In [4]:
from xlmexlab.nanoparticle_paragraph import NanoparticleExtractor

_nano_p = NanoparticleExtractor()

text = """ The micelles were prepared via the solvent displacement method.44 The DLS analysis revealed hydrodynamic diameters (Table 2) ranging from 37\u00b14 nm (OPDMA_9\u2013b\u2013PCL_40) to 100\u00a0\u00b1\u00a06 nm (OPDMA_10\u2013b\u2013PCL_9) , with zeta potentials maintaining near-neutral values across all formulations, consistent with the zwitterionic surface characteristics. The reduction in particle size with increasing hydrophobic chain length may be attributed to a lower critical micelle concentration45 (CMC) (Figure S4) and tighter packing of hydrophobic segments [46-48] The OPDMA_19\u2013b\u2013PCL_61 formu- lation (hereinafter simplified as OPDMA-PCL) was selected for subsequent biological studies based on its optimal size profile (51\u0305\u00b12\u00a0nm,P\u0305\u0305D\u0305\u0305I\u0305=0.088) . This size range facilitates both prolonged blood circulation (avoiding renal clearance threshold <8nm^49) and efficient tumor penetration via the EPR effect. The TEM images reveal that the micelles are spherical in shape (Figure 2a). Meanwhile, the micelles can maintain the stability of particle size and zeta potential in different solutions and have long-term stability, which is conducive to long-term storage in practical life (Figures 2b and S6). The zeta potential of micelles exhibited pH-dependent behavior when they were dispersed in PBS at varying pH levels (Figure 2c). Within the acidic range (pH 2\u22124), the zeta potential progressively decreased as pH increased. In contrast, minimal variation in zeta potential was observed under neutral to alkaline conditions (pH 6\u221212). Traditional zwitterionic polymers, including poly(carboxybetaine), poly(sulfobetaine), and poly(phosphorylcholine), typically exhibit isoelectric point (IEP) values of 6, 5.5\u22126.5, and 7.5, respectively.50\u221252 The zeta potential approached 0 mV at around pH 4.93, indicating that this pH corresponds to the IEP of the polymer, which broadens the isoelectric range of the zwitterionic polymer."""

print (_nano_p._extract_pdi(text))

False


In [6]:
import json
import re

def corta_a_meio(texto: str) -> bool:
    """True se o texto parece cortado a meio de frase: termina em letra minúscula."""
    texto = texto.rstrip()
    if not texto:
        return False
    return bool(re.search(r'[a-z]$', texto))

def fundir_paragrafos(blocks, log):
    i = 0
    while i < len(blocks) - 1:
        atual = blocks[i]
        seguinte = blocks[i + 1]

        if (atual.get("type") == "paragraph"
                and seguinte.get("type") == "paragraph"
                and atual.get("content", "").strip()
                and seguinte.get("content", "").strip()
                and corta_a_meio(atual["content"])):

            sep = "" if atual["content"].rstrip().endswith("-") else " "
            novo_content = atual["content"].rstrip() + sep + seguinte["content"].lstrip()

            log.append({
                "index_atual": i,
                "index_seguinte": i + 1,
                "fim_atual": atual["content"][-80:],
                "inicio_seguinte": seguinte["content"][:80],
            })

            atual["content"] = novo_content
            seguinte["content"] = ""
            # não avança i, para permitir fusões em cadeia
        else:
            i += 1
    return blocks

if __name__ == "__main__":
    in_path = "/Users/leagabay/Downloads/result_mineruvlm/10.1016_j.actbio.2017.08.034/10.1016_j.actbio.2017.08.034_content.json"
    out_path = "/Users/leagabay/Downloads/result_mineruvlm/10.1016_j.actbio.2017.08.034/10.1016_j.actbio.2017.08.034_content_output.json"

    with open(in_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    log = []
    data["blocks"] = fundir_paragrafos(data["blocks"], log)

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Total de fusões: {len(log)}")
    for entry in log:
        print("---")
        print(f"[{entry['index_atual']}] ...{entry['fim_atual']!r}")
        print(f"[{entry['index_seguinte']}] {entry['inicio_seguinte']!r}...")

Total de fusões: 14
---
[3] ...'CrossMark'
[4] 'Yang Fan a, Qingjie Wang b,c, Guimei Lin a,d,, Yanbin Shi e, Zili Gu a, Tingting'...
---
[5] ...'^aSchool of Pharmaceutical Science, Shandong University, Jinan 250012, China'
[6] '^bInstitute of Basic Medical Sciences, Qilu Hospital, Shandong University, Jinan'...
---
[7] ...'nese Ministry of Health, Qilu Hospital, Shandong University, Jinan 250012, China'
[8] '^d Key Laboratory of Chemical Biology (Ministry of Education), School of Pharmac'...
---
[18] ...'CD44-oriented tumor targeting'
[19] 'Prodrug-modified liposome nanocomplexes'...
---
[20] ...'Triple-negative breast cancer'
[21] 'Potentiating effect'...
---
[31] ...'yield the HA-GEM and DTX-CL composite nanocomplexes (Combo NCs). The DTX-CLs can'
[32] "hydrophobilize' HA-GEM via electrostatic attraction and increase the bioavailabi"...
---
[38] ...'lines (MCF-7 and MDAMB-231) were kindly provided by the School of Pharmaceutical'
[39] 'Science and the School of Pharmacology, Shandong

In [2]:
from xlmexlab.nanoparticle_paragraph import NanoparticleExtractor

_nanoparticles_pargraph = NanoparticleExtractor()

text = """
to obtain hollow microbubbles; third, blowing SF_6 and re- dissolving the microbubbles with shook to push the gas diffusing into MBs to obtain PTX@RGD-MBs.
27 Due to the hydrophilic of PEG, the PTX@RGD-MBs was uniformly dis- solved in the solution, along with spherical appearance and smooth surface without conjugation (Fig. 2B). 
The size and zeta potential of PTX@RGD-MBs were 1741.67±67.72:nm (Fig. 2C) and -0.665 ± 0.033 mV (Fig. 2D) respectively. 
The poly- dispersity index (PDI) was 0.106. The PTX@RGD-MBs exhibited an excellent drug encapsulation efficiency (EE%) of 91.07% and a drug loading (DL%) of 4.01% for PTX (Table 1), 
suggesting the prevention from side effects created by unbound chemothera- peutic drugs.
"""

print (_nanoparticles_pargraph._extract_size(text))

[<re.Match object; span=(337, 341), match='size'>]
True
